In [9]:
######### Plot violin for Landsat signature across three land cover types ########
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import os
import pandas as pd

In [10]:
#### New script ####
# Root folder
root_folder = "/Volumes/MaloneLab/Research/Whitezone/Data/KML/"

# Target years
years = [1985, 1994, 1999, 2003, 2009, 2013, 2018, 2023]

# Dictionary to store DataFrames
data_dict = {}

for year in years:
    year_folder = os.path.join(root_folder, str(year))
    csv_path = os.path.join(year_folder, "median.csv")
    
    if os.path.exists(csv_path):
        print(f"Reading: {csv_path}")
        df = pd.read_csv(csv_path)
        data_dict[year] = df
    else:
        print(f"⚠️ median.csv not found for {year}")

Reading: /Volumes/MaloneLab/Research/Whitezone/Data/KML/1985/median.csv
Reading: /Volumes/MaloneLab/Research/Whitezone/Data/KML/1994/median.csv
Reading: /Volumes/MaloneLab/Research/Whitezone/Data/KML/1999/median.csv
Reading: /Volumes/MaloneLab/Research/Whitezone/Data/KML/2003/median.csv
Reading: /Volumes/MaloneLab/Research/Whitezone/Data/KML/2009/median.csv
Reading: /Volumes/MaloneLab/Research/Whitezone/Data/KML/2013/median.csv
Reading: /Volumes/MaloneLab/Research/Whitezone/Data/KML/2018/median.csv
Reading: /Volumes/MaloneLab/Research/Whitezone/Data/KML/2023/median.csv


In [25]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

# === Settings ===
root_folder = "/Volumes/MaloneLab/Research/Whitezone/Data/KML/"
years = [1985, 1994, 1999, 2003, 2009, 2013, 2018, 2023]

land_cover_order = ["WZ", "TZ", "VZ"]
land_cover_colors = {
    "WZ": "#b08b5b",
    "TZ": "#a1d996",
    "VZ": "#00441b"
}

spectral_cols = ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"]
index_cols = ["ndvi", "ndmi", "savi"]  # Keep this order for the plot
all_features = spectral_cols + index_cols

# === Custom x-axis labels ===
feature_labels = [
    "Blue", "Green", "Red", "Near Infrared", 
    "Shortwave Infrared 1", "Shortwave Infrared 2", 
    "NDVI", "NDMI", "SAVI"
]

# === Read and combine ===
df_list = []
for year in years:
    csv_path = os.path.join(root_folder, str(year), "median.csv")
    if os.path.exists(csv_path):
        df_year = pd.read_csv(csv_path)
        df_year["Year"] = year
        df_list.append(df_year)
    else:
        print(f"⚠️ median.csv not found for {year}")

df_all = pd.concat(df_list, ignore_index=True)

# === Filter out MZ, keep only WZ/TZ/VZ ===
df_all = df_all[df_all["descriptio"].isin(land_cover_order)]

# === Reverse NDVI and NDMI columns ===
if 'ndvi' in df_all.columns and 'ndmi' in df_all.columns:
    df_all[['ndvi', 'ndmi']] = df_all[['ndmi', 'ndvi']]
    print("✅ Swapped NDVI and NDMI columns in df_all")

# === Normalize spectral columns ===
scaler = MinMaxScaler()
df_all[spectral_cols] = scaler.fit_transform(df_all[spectral_cols])

# === Melt to long format ===
melted = pd.melt(
    df_all,
    id_vars=["descriptio", "Year"],
    value_vars=all_features,
    var_name="Feature",
    value_name="Value"
)

# === Trim to 10–90th percentile per group ===
def trim_percentiles(group):
    lower = group["Value"].quantile(0.2)
    upper = group["Value"].quantile(0.8)
    return group[(group["Value"] >= lower) & (group["Value"] <= upper)]

trimmed = melted.groupby(["Feature", "descriptio"], group_keys=False).apply(trim_percentiles)

# === Plot ===
plt.figure(figsize=(14, 6))
sns.set(style="whitegrid", font="Times New Roman")

sns.violinplot(
    data=trimmed,
    x="Feature",
    y="Value",
    hue="descriptio",
    hue_order=land_cover_order,
    palette=land_cover_colors,
    inner="quartile",
    split=False
)

plt.xlabel("")
plt.ylabel("Value", fontsize=16)

# Apply custom x-axis labels
plt.xticks(
    ticks=range(len(feature_labels)),
    labels=feature_labels,
    rotation=45,
    fontsize=16,
    fontname="Times New Roman"
)

plt.yticks(fontsize=16, fontname="Times New Roman")

# Create custom legend labels (WZ -> DZ)
legend_labels = ['DZ', 'TZ', 'VZ']
handles, _ = plt.gca().get_legend_handles_labels()
plt.legend(
    handles=handles,
    labels=legend_labels,
    title="Land Cover",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    fontsize=16,
    title_fontsize=16
)

plt.tight_layout()

# Save the figure
out_path = "/Users/bz294/Documents/WhiteZone/Figure/land_cover_distribution_trimmed.tiff"
plt.savefig(out_path, format="tiff", dpi=300, bbox_inches="tight")
plt.show()

print(f"✅ Plot (10–90 percentile) saved to {out_path}")

✅ Swapped NDVI and NDMI columns in df_all


/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_78374/2529078927.py:70: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  trimmed = melted.groupby(["Feature", "descriptio"], group_keys=False).apply(trim_percentiles)


✅ Plot (10–90 percentile) saved to /Users/bz294/Documents/WhiteZone/Figure/land_cover_distribution_trimmed.tiff


/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_78374/2529078927.py:119: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
